In [1]:
%reset -f

import numpy as np 
import pandas as pd 
import yfinance as yf
import matplotlib.pyplot as plt 
    

In [ ]:
tickers = ["RY.TO", "BNS.TO", "TD.TO", "BMO.TO"]
start_date = "2000-01-01"
end_date = "2026-02-28"

plt.figure(figsize=(12,5))

for ticker in tickers:
    df = yf.download(ticker, start=start_date, end=end_date, interval="1wk", auto_adjust=True)

    prices = df["Close"].dropna()
    log_returns = np.log(prices / prices.shift(1)).dropna()

    signal = log_returns - log_returns.mean()

    n = len(signal)
    fft_vals = np.fft.fft(signal.values)
    freq = np.fft.fftfreq(n, d= 1)

    mask = freq > 0 
    freq = freq[mask]
    fft_vals = fft_vals[mask]

    magnitudes = np.abs(fft_vals)

    magnitudes = magnitudes / np.max(magnitudes)

    period = 1 / freq

    valid = (period >= 2) & (period <= 52)

    plt.plot(period[valid], magnitudes[valid], label = ticker )

plt.xlabel("Period (weeks)")
plt.ylabel("Normalized Magnitude")
plt.title("Frequency Spectrum Comparison (Weekly Returns)")
plt.legend()
plt.grid()
plt.show()



In [ ]:
window = [
    ("2000-01-01", "2005-01-01"),
    ("2005-01-01","2010-01-01"),
    ("2010-01-01", "2015-01-01"),
    ("2015-01-01", "2020-01-01"),
    ("2020-01-01", "2025-01-01"),
]

num_windows = len(window)

plt.figure(figsize=(12,8))

RY = "RY.TO"

for i, (start, end) in enumerate(window, 1):

    df_RY = yf.download(RY, start=start, end=end, interval="1wk", auto_adjust=True)

    prices_RY = df_RY["Close"].dropna()
    log_returns_RY = np.log(prices_RY / prices_RY.shift(1)).dropna()

    signal_RY = log_returns_RY - log_returns_RY.mean()

    n_RY = len(signal_RY)
    fft_valsRY = np.fft.fft(signal_RY.values)
    freqs_RY = np.fft.fftfreq(n_RY, d=1)

    mask_RY = freqs_RY > 0
    freqs_RY = freqs_RY[mask_RY]
    fft_valsRY = fft_valsRY[mask_RY]

    magnitudes_RY = np.abs(fft_valsRY)
    magnitudes_RY = magnitudes_RY / np.max(magnitudes_RY)

    periods_RY = 1/freqs_RY

    valid_RY = (periods_RY >= 2) & (periods_RY <= 52)

    plt.subplot(num_windows, 1, i)
    plt.plot(periods_RY[valid_RY], magnitudes_RY[valid_RY])
    plt.title(f"{RY} FFT ({start[:4]}-{end[:4]})")
    plt.xlabel("Period (weeks)")
    plt.ylabel("Nor. Mag.")
    plt.grid()

plt.tight_layout()
plt.show()



In [ ]:
#output of both FFT analyses shows that there is little to no cyclicality in the major Canadian Banks
#Most peaks in the 5-10 week range are most likely the result of noise rather than actual cyclicality.
#Log normal returns are not cyclical at least for RBC since there is no consistent cycle over 5 year periods from 2000-2025. 